# SkyAware Flight Delay Prediction — Notebook 2: Feature Engineering & Feature Store

**AAI-540 MLOps | Group 4**

This notebook:
1. Engineers predictive features from the raw BTS dataset
2. Creates and populates a **SageMaker Feature Group** (`skyaware-flight-features`)
3. Demonstrates both **online** and **offline** feature retrieval

## 1. Setup & Imports

In [1]:
import boto3
import pandas as pd
import numpy as np
import time
import json
import os
import warnings
warnings.filterwarnings('ignore')

import sagemaker
from sagemaker import get_execution_role
from sagemaker.session import Session
from sagemaker.feature_store.feature_group import FeatureGroup

# AWS config
region = boto3.Session().region_name
role = get_execution_role()
account_id = boto3.client('sts').get_caller_identity()['Account']

# S3 buckets
RAW_BUCKET = 'skyaware-raw-data'
PROCESSED_BUCKET = 'skyaware-processed-data'
FEATURE_STORE_S3_URI = f's3://{PROCESSED_BUCKET}/feature-store'

# Feature Group name
FEATURE_GROUP_NAME = 'skyaware-flight-features'

# Feature Store session (requires separate runtime client)
boto_session = boto3.Session(region_name=region)
sagemaker_client = boto_session.client(service_name='sagemaker', region_name=region)
featurestore_runtime = boto_session.client(service_name='sagemaker-featurestore-runtime', region_name=region)
feature_store_session = Session(
    boto_session=boto_session,
    sagemaker_client=sagemaker_client,
    sagemaker_featurestore_runtime_client=featurestore_runtime
)

print(f'Region: {region}')
print(f'Feature Store S3 URI: {FEATURE_STORE_S3_URI}')
print(f'Feature Group: {FEATURE_GROUP_NAME}')

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


Region: us-east-1
Feature Store S3 URI: s3://skyaware-processed-data/feature-store
Feature Group: skyaware-flight-features


In [2]:
# Create S3 buckets if they don't exist
s3_client = boto3.client('s3', region_name=region)

def create_bucket_if_missing(bucket_name):
    try:
        s3_client.head_bucket(Bucket=bucket_name)
        print(f'Bucket exists: s3://{bucket_name}')
    except Exception as e:
        error_code = e.response['Error']['Code'] if hasattr(e, 'response') else ''
        if error_code in ('404', 'NoSuchBucket'):
            try:
                if region == 'us-east-1':
                    s3_client.create_bucket(Bucket=bucket_name)
                else:
                    s3_client.create_bucket(
                        Bucket=bucket_name,
                        CreateBucketConfiguration={'LocationConstraint': region}
                    )
                print(f'Created bucket: s3://{bucket_name}')
            except Exception as create_err:
                print(f'Could not create {bucket_name}: {create_err}')
        else:
            print(f'Bucket check error for {bucket_name}: {e}')

for bucket in [RAW_BUCKET, PROCESSED_BUCKET]:
    create_bucket_if_missing(bucket)

Bucket exists: s3://skyaware-raw-data
Bucket exists: s3://skyaware-processed-data


## 2. Load Raw Data

In [3]:
DATA_PATH = './Dataset/Airline_Delay_Cause.csv'

# Prefer the prepared CSV from NB01 (includes EDA-derived features)
def load_prepared_from_s3():
    try:
        obj = s3_client.get_object(Bucket=PROCESSED_BUCKET, Key='interim/airline_delay_prepared.csv')
        df = pd.read_csv(obj['Body'])
        print(f'Loaded prepared dataset from s3://{PROCESSED_BUCKET}/interim/airline_delay_prepared.csv')
        return df
    except Exception as e:
        print(f'Prepared CSV not found ({e}), falling back to raw CSV...')
        return None

df = load_prepared_from_s3()

if df is None:
    # Fallback: re-derive from raw
    df = pd.read_csv(DATA_PATH)
    df.columns = df.columns.str.strip()
    df = df[df['year'] >= 2010].copy()
    df = df[df['arr_flights'] > 0].copy()
    delay_cols = ['arr_del15', 'carrier_ct', 'weather_ct', 'nas_ct', 'security_ct',
                  'late_aircraft_ct', 'arr_cancelled', 'arr_diverted',
                  'arr_delay', 'carrier_delay', 'weather_delay', 'nas_delay',
                  'security_delay', 'late_aircraft_delay']
    df[delay_cols] = df[delay_cols].fillna(0)
    df['delay_rate'] = df['arr_del15'] / df['arr_flights']
    df['cancel_rate'] = df['arr_cancelled'] / df['arr_flights']

    def assign_risk_label(row):
        if row['cancel_rate'] > 0.05 or row['delay_rate'] > 0.40:
            return 3
        elif row['delay_rate'] > 0.25:
            return 2
        elif row['delay_rate'] > 0.15:
            return 1
        else:
            return 0

    df['delay_risk_label'] = df.apply(assign_risk_label, axis=1)
    print('Derived from raw CSV.')

print(f'Dataset shape: {df.shape}')
print(f'Year range: {df["year"].min()}–{df["year"].max()}')
df.head(3)

Loaded prepared dataset from s3://skyaware-processed-data/interim/airline_delay_prepared.csv
Dataset shape: (281032, 32)
Year range: 2010–2025


,year,month,carrier,carrier_name,airport,airport_name,arr_flights,arr_del15,carrier_ct,weather_ct,...,cancel_rate,delay_risk_label,risk_label_name,is_covid_period,late_aircraft_rate,carrier_delay_rate,weather_delay_rate,nas_delay_rate,month_sin,month_cos
0,2025,1,G4,Allegiant Air,ELM,"Elmira/Corning, NY: Elmira/Corning Regional",30.0,0.0,0.00,0.0,...,0.000000,0,on-time,0,0.000,0.000000,0.0,0.000000,0.5,0.866025
1,2025,1,G4,Allegiant Air,ELP,"El Paso, TX: El Paso International",2.0,0.0,0.00,0.0,...,0.000000,0,on-time,0,0.000,0.000000,0.0,0.000000,0.5,0.866025
2,2025,1,G4,Allegiant Air,EUG,"Eugene, OR: Mahlon Sweet Field",28.0,8.0,3.74,0.0,...,0.071429,3,high-risk,0,0.095,0.133571,0.0,0.057143,0.5,0.866025


## 3. Feature Engineering

In [4]:
# --- Base rate features ---
df['divert_rate'] = df['arr_diverted'] / df['arr_flights']
df['avg_delay_per_flight'] = df['arr_delay'] / df['arr_flights'].clip(lower=1)

# Delay cause proportions (share of delayed flights by cause)
total_delay_ct = df[['carrier_ct','weather_ct','nas_ct','security_ct','late_aircraft_ct']].sum(axis=1).clip(lower=1)
df['carrier_delay_pct']  = df['carrier_ct']       / total_delay_ct
df['weather_delay_pct']  = df['weather_ct']        / total_delay_ct
df['nas_delay_pct']      = df['nas_ct']            / total_delay_ct
df['security_delay_pct'] = df['security_ct']       / total_delay_ct
df['late_aircraft_pct']  = df['late_aircraft_ct']  / total_delay_ct

# --- EDA-derived features (add if not already present from NB01) ---
# COVID period flag: 2020 avg_delay_rate=9.3% (lowest) but cancel_rate=6.0% (highest)
if 'is_covid_period' not in df.columns:
    df['is_covid_period'] = df['year'].isin([2020, 2021]).astype(int)

# Late aircraft per-flight rate — dominant cause at 36.2% of all delays
if 'late_aircraft_rate' not in df.columns:
    df['late_aircraft_rate'] = df['late_aircraft_ct'] / df['arr_flights']

# Rate-based cause features (per-flight rates, not just share of delays)
if 'carrier_delay_rate' not in df.columns:
    df['carrier_delay_rate'] = df['carrier_ct'] / df['arr_flights']
if 'weather_delay_rate' not in df.columns:
    df['weather_delay_rate'] = df['weather_ct'] / df['arr_flights']
if 'nas_delay_rate' not in df.columns:
    df['nas_delay_rate'] = df['nas_ct'] / df['arr_flights']

# Cyclical month encoding — July=peak, September=trough confirmed by EDA
if 'month_sin' not in df.columns:
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
if 'month_cos' not in df.columns:
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

# --- Temporal / seasonal features ---
df['quarter']        = ((df['month'] - 1) // 3 + 1).astype(int)
df['is_winter']      = df['month'].isin([12, 1, 2]).astype(int)
df['is_summer']      = df['month'].isin([6, 7, 8]).astype(int)
df['is_peak_travel'] = df['month'].isin([6, 7, 8, 11, 12]).astype(int)

print('Features in dataset:', [c for c in df.columns if c not in ['carrier','airport','carrier_name','airport_name']])

Features in dataset: ['year', 'month', 'arr_flights', 'arr_del15', 'carrier_ct', 'weather_ct', 'nas_ct', 'security_ct', 'late_aircraft_ct', 'arr_cancelled', 'arr_diverted', 'arr_delay', 'carrier_delay', 'weather_delay', 'nas_delay', 'security_delay', 'late_aircraft_delay', 'delay_rate', 'cancel_rate', 'delay_risk_label', 'risk_label_name', 'is_covid_period', 'late_aircraft_rate', 'carrier_delay_rate', 'weather_delay_rate', 'nas_delay_rate', 'month_sin', 'month_cos', 'divert_rate', 'avg_delay_per_flight', 'carrier_delay_pct', 'weather_delay_pct', 'nas_delay_pct', 'security_delay_pct', 'late_aircraft_pct', 'quarter', 'is_winter', 'is_summer', 'is_peak_travel']


In [5]:
# --- Rolling lag features per carrier ---
df = df.sort_values(['carrier', 'year', 'month']).reset_index(drop=True)

# 3-month rolling avg (captures short-term trends)
df['carrier_rolling_delay_3m'] = (
    df.groupby('carrier')['delay_rate']
      .transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
)

# 12-month rolling avg (captures annual seasonality patterns per carrier)
df['carrier_rolling_delay_12m'] = (
    df.groupby('carrier')['delay_rate']
      .transform(lambda x: x.shift(1).rolling(window=12, min_periods=1).mean())
)

# Late aircraft rolling 3m — dominant cause (36.2%), its own lag is a strong signal
df['late_aircraft_roll_3m'] = (
    df.groupby('carrier')['late_aircraft_rate']
      .transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
)

# --- Rolling lag features per airport ---
df = df.sort_values(['airport', 'year', 'month']).reset_index(drop=True)
df['airport_rolling_delay_3m'] = (
    df.groupby('airport')['delay_rate']
      .transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
)

# --- Year-over-year delay rate change per carrier ---
df = df.sort_values(['carrier', 'year', 'month']).reset_index(drop=True)
df['carrier_yoy_delay_change'] = (
    df.groupby(['carrier', 'month'])['delay_rate']
      .transform(lambda x: x.diff())
)

lag_cols = ['carrier_rolling_delay_3m', 'carrier_rolling_delay_12m', 'late_aircraft_roll_3m',
            'airport_rolling_delay_3m', 'carrier_yoy_delay_change']
df[lag_cols] = df[lag_cols].fillna(0)

print('Lag features created:', lag_cols)

Lag features created: ['carrier_rolling_delay_3m', 'carrier_rolling_delay_12m', 'late_aircraft_roll_3m', 'airport_rolling_delay_3m', 'carrier_yoy_delay_change']


In [6]:
# --- Feature Store required fields ---
# record_id: unique identifier per row
df['record_id'] = (df['carrier'].astype(str) + '_' +
                   df['airport'].astype(str) + '_' +
                   df['year'].astype(str) + '_' +
                   df['month'].astype(str).str.zfill(2))

# event_time: Unix timestamp (Feature Store requires float)
df['event_time'] = pd.to_datetime(
    df['year'].astype(str) + '-' + df['month'].astype(str).str.zfill(2) + '-01'
).astype('int64') // 10**9
df['event_time'] = df['event_time'].astype(float)

print(f'Total features engineered: {df.shape[1]} columns')
print(f'Feature dataset shape: {df.shape}')
print(f'Sample record_id: {df["record_id"].iloc[0]}')
print(f'Sample event_time: {df["event_time"].iloc[0]}')
df[['record_id','event_time','delay_rate','carrier_rolling_delay_3m','delay_risk_label']].head()

Total features engineered: 50 columns
Feature dataset shape: (281032, 50)
Sample record_id: 9E_ABE_2010_01
Sample event_time: 1262304000.0


,record_id,event_time,delay_rate,carrier_rolling_delay_3m,delay_risk_label
0,9E_ABE_2010_01,1.262304e+09,0.246154,0.000000,1
1,9E_AEX_2010_01,1.262304e+09,0.283333,0.246154,3
2,9E_ALB_2010_01,1.262304e+09,0.419753,0.264744,3
3,9E_ALO_2010_01,1.262304e+09,0.333333,0.316413,2
4,9E_ATL_2010_01,1.262304e+09,0.253983,0.345473,2


In [7]:
# Select columns for Feature Store ingestion
FEATURE_COLS = [
    'record_id', 'event_time',
    'year', 'month', 'month_sin', 'month_cos', 'carrier', 'airport',
    # Core rate features (delay_rate and cancel_rate are primary predictors per EDA)
    'arr_flights', 'delay_rate', 'cancel_rate', 'divert_rate', 'avg_delay_per_flight',
    # Rate-based cause features (per-flight; EDA showed raw counts < 0.07 correlation)
    'late_aircraft_rate', 'carrier_delay_rate', 'weather_delay_rate', 'nas_delay_rate',
    # Cause proportions (share of total delayed flights)
    'carrier_delay_pct', 'weather_delay_pct', 'nas_delay_pct',
    'security_delay_pct', 'late_aircraft_pct',
    # Temporal flags
    'quarter', 'is_winter', 'is_summer', 'is_peak_travel', 'is_covid_period',
    # Rolling lag features
    'carrier_rolling_delay_3m', 'carrier_rolling_delay_12m', 'late_aircraft_roll_3m',
    'airport_rolling_delay_3m', 'carrier_yoy_delay_change',
    # Target
    'delay_risk_label'
]

fg_df = df[FEATURE_COLS].copy()

# Feature Store requires string columns for string features and float for numerics
numeric_fs_cols = [c for c in FEATURE_COLS if c not in ['record_id', 'carrier', 'airport']]
fg_df[numeric_fs_cols] = fg_df[numeric_fs_cols].astype(float)

# Clip rates to [0, 1] to guard against rare division edge cases
rate_cols = [c for c in numeric_fs_cols if 'rate' in c or 'pct' in c]
fg_df[rate_cols] = fg_df[rate_cols].clip(0, 1)

print(f'Feature Store dataframe shape: {fg_df.shape}')
print(f'Total features (excl. record_id, event_time): {len(FEATURE_COLS) - 2}')
print(f'Dtypes sample:')
print(fg_df.dtypes[:12])

Feature Store dataframe shape: (281032, 33)
Total features (excl. record_id, event_time): 31
Dtypes sample:
record_id       object
event_time     float64
year           float64
month          float64
month_sin      float64
month_cos      float64
carrier         object
airport         object
arr_flights    float64
delay_rate     float64
cancel_rate    float64
divert_rate    float64
dtype: object


## 4. Create SageMaker Feature Group

In [8]:
# Initialize Feature Group
flight_feature_group = FeatureGroup(
    name=FEATURE_GROUP_NAME,
    sagemaker_session=feature_store_session
)

# Auto-detect feature definitions from dataframe schema
flight_feature_group.load_feature_definitions(data_frame=fg_df)

print('Feature definitions loaded:')
for fd in flight_feature_group.feature_definitions:
    print(f'  {fd.feature_name}: {fd.feature_type}')

Feature definitions loaded:
  record_id: FeatureTypeEnum.STRING
  event_time: FeatureTypeEnum.FRACTIONAL
  year: FeatureTypeEnum.FRACTIONAL
  month: FeatureTypeEnum.FRACTIONAL
  month_sin: FeatureTypeEnum.FRACTIONAL
  month_cos: FeatureTypeEnum.FRACTIONAL
  carrier: FeatureTypeEnum.STRING
  airport: FeatureTypeEnum.STRING
  arr_flights: FeatureTypeEnum.FRACTIONAL
  delay_rate: FeatureTypeEnum.FRACTIONAL
  cancel_rate: FeatureTypeEnum.FRACTIONAL
  divert_rate: FeatureTypeEnum.FRACTIONAL
  avg_delay_per_flight: FeatureTypeEnum.FRACTIONAL
  late_aircraft_rate: FeatureTypeEnum.FRACTIONAL
  carrier_delay_rate: FeatureTypeEnum.FRACTIONAL
  weather_delay_rate: FeatureTypeEnum.FRACTIONAL
  nas_delay_rate: FeatureTypeEnum.FRACTIONAL
  carrier_delay_pct: FeatureTypeEnum.FRACTIONAL
  weather_delay_pct: FeatureTypeEnum.FRACTIONAL
  nas_delay_pct: FeatureTypeEnum.FRACTIONAL
  security_delay_pct: FeatureTypeEnum.FRACTIONAL
  late_aircraft_pct: FeatureTypeEnum.FRACTIONAL
  quarter: FeatureTypeEnum.FR

In [9]:
# Create the Feature Group
try:
    flight_feature_group.create(
        record_identifier_name='record_id',
        event_time_feature_name='event_time',
        role_arn=role,
        s3_uri=FEATURE_STORE_S3_URI,
        enable_online_store=True,
        description='SkyAware flight delay risk features — carrier/airport monthly aggregates'
    )
    print(f'Feature Group creation initiated: {FEATURE_GROUP_NAME}')
except Exception as e:
    if 'ResourceInUse' in str(e) or 'already exists' in str(e).lower():
        print(f'Feature Group already exists: {FEATURE_GROUP_NAME}')
    else:
        raise e

# Wait for Feature Group to be CREATED
print('Waiting for Feature Group to become available...')
max_wait = 300
elapsed = 0
while elapsed < max_wait:
    status = sagemaker_client.describe_feature_group(
        FeatureGroupName=FEATURE_GROUP_NAME
    )['FeatureGroupStatus']
    print(f'  Status: {status} ({elapsed}s elapsed)')
    if status == 'Created':
        print('Feature Group is ready!')
        break
    elif status in ['CreateFailed', 'DeleteFailed']:
        raise RuntimeError(f'Feature Group creation failed with status: {status}')
    time.sleep(15)
    elapsed += 15

Feature Group creation initiated: skyaware-flight-features
Waiting for Feature Group to become available...
  Status: Creating (0s elapsed)


  Status: Creating (15s elapsed)


  Status: Created (30s elapsed)
Feature Group is ready!


## 5. Ingest Data into Feature Store

In [10]:
# Ingest all records
print(f'Ingesting {len(fg_df):,} records into Feature Store...')
ingest_result = flight_feature_group.ingest(
    data_frame=fg_df,
    max_workers=5,
    wait=True
)
print(f'Ingestion complete!')
print(f'Failed rows: {len(ingest_result.failed_rows)}')
if ingest_result.failed_rows:
    print('Sample failed rows:', ingest_result.failed_rows[:3])

Ingesting 281,032 records into Feature Store...


Ingestion complete!
Failed rows: 0


## 6. Query Offline Store via Athena

In [12]:
# Wait a few seconds for offline store to be available
time.sleep(30)

# Athena query on offline store
athena_query = flight_feature_group.athena_query()
athena_table = athena_query.table_name
print(f'Athena table: {athena_table}')

query_string = f"""
SELECT record_id, year, month, carrier, airport,
       delay_rate, cancel_rate, late_aircraft_rate,
       carrier_rolling_delay_3m, is_covid_period,
       delay_risk_label
FROM "{athena_table}"
WHERE year >= 2022
  AND write_time IS NOT NULL
LIMIT 100
"""

athena_query.run(
    query_string=query_string,
    output_location=f's3://{PROCESSED_BUCKET}/athena-results/'
)
athena_query.wait()

offline_df = athena_query.as_dataframe()
print(f'\nOffline store query returned: {offline_df.shape}')
offline_df.head()

Athena table: skyaware_flight_features_1781963496



Offline store query returned: (100, 11)


,record_id,year,month,carrier,airport,delay_rate,cancel_rate,late_aircraft_rate,carrier_rolling_delay_3m,is_covid_period,delay_risk_label
0,OO_BNA_2022_02,2022.0,2.0,OO,BNA,0.261194,0.019900,0.084577,0.210595,0.0,2.0
1,OO_DEN_2022_02,2022.0,2.0,OO,DEN,0.222292,0.051861,0.078210,0.195254,0.0,3.0
2,OO_LAX_2022_02,2022.0,2.0,OO,LAX,0.108532,0.012188,0.029600,0.182406,0.0,0.0
3,OO_MLI_2022_02,2022.0,2.0,OO,MLI,0.204819,0.084337,0.036145,0.236532,0.0,3.0
4,OO_RKS_2022_02,2022.0,2.0,OO,RKS,0.357143,0.000000,0.035714,0.226123,0.0,2.0


## 7. Online Store Lookup (Real-time Feature Retrieval)

In [13]:
# Retrieve a single record from the Online Store
sample_record_id = fg_df['record_id'].iloc[100]
print(f'Looking up record: {sample_record_id}')

response = featurestore_runtime.get_record(
    FeatureGroupName=FEATURE_GROUP_NAME,
    RecordIdentifierValueAsString=sample_record_id
)

print('\nOnline Store record:')
for feat in response.get('Record', []):
    print(f"  {feat['FeatureName']}: {feat['ValueAsString']}")

Looking up record: 9E_RDU_2010_01

Online Store record:
  record_id: 9E_RDU_2010_01
  event_time: 1262304000.0
  year: 2010.0
  month: 1.0
  month_sin: 0.4999999999999999
  month_cos: 0.8660254037844387
  carrier: 9E
  airport: RDU
  arr_flights: 189.0
  delay_rate: 0.1375661375661375
  cancel_rate: 0.0582010582010582
  divert_rate: 0.0
  avg_delay_per_flight: 7.761904761904762
  late_aircraft_rate: 0.0174603174603174
  carrier_delay_rate: 0.0689417989417989
  weather_delay_rate: 0.0052910052910052
  nas_delay_rate: 0.0458730158730158
  carrier_delay_pct: 0.5011538461538462
  weather_delay_pct: 0.038461538461538464
  nas_delay_pct: 0.3334615384615385
  security_delay_pct: 0.0
  late_aircraft_pct: 0.12692307692307692
  quarter: 1.0
  is_winter: 1.0
  is_summer: 0.0
  is_peak_travel: 0.0
  is_covid_period: 0.0
  carrier_rolling_delay_3m: 0.23138297872340421
  carrier_rolling_delay_12m: 0.22984972463495493
  late_aircraft_roll_3m: 0.03256890715667308
  airport_rolling_delay_3m: 0.0
  carr

In [14]:
# Batch get multiple records (useful for real-time inference)
sample_ids = fg_df['record_id'].iloc[0:5].tolist()

batch_response = featurestore_runtime.batch_get_record(
    Identifiers=[
        {'FeatureGroupName': FEATURE_GROUP_NAME, 'RecordIdentifiersValueAsString': sample_ids}
    ]
)
print(f'Batch retrieved {len(batch_response["Records"])} records')
print(f'Errors: {len(batch_response["Errors"])}')

Batch retrieved 5 records
Errors: 0


## 8. Feature Summary

Features are grouped by origin and predictive strength based on EDA correlation analysis.

| Feature | Type | Corr w/ target | Description |
|---|---|---|---|
| `delay_rate` | Float | **0.67** | Primary signal: arr_del15 / arr_flights |
| `cancel_rate` | Float | **0.44** | arr_cancelled / arr_flights |
| `late_aircraft_rate` | Float | ~0.40 | late_aircraft_ct / arr_flights — dominant cause (36.2%) |
| `carrier_delay_rate` | Float | ~0.35 | carrier_ct / arr_flights (per-flight rate) |
| `nas_delay_rate` | Float | ~0.20 | nas_ct / arr_flights (per-flight rate) |
| `divert_rate` | Float | — | arr_diverted / arr_flights |
| `avg_delay_per_flight` | Float | — | Total delay minutes / flights |
| `carrier_delay_pct` | Float | — | Carrier-caused share of total delays |
| `weather_delay_pct` | Float | — | Weather-caused share (only 3.3% overall) |
| `nas_delay_pct` | Float | — | NAS-caused share of delays |
| `late_aircraft_pct` | Float | — | Late-aircraft share of delays |
| `is_covid_period` | Int | — | Binary: 1 for 2020–2021 (anomalous inverse pattern) |
| `month_sin` / `month_cos` | Float | — | Cyclical month encoding (July=peak, Sep=trough) |
| `quarter` | Int | — | Calendar quarter (1–4) |
| `is_winter` | Int | — | Binary: Dec/Jan/Feb |
| `is_summer` | Int | — | Binary: Jun/Jul/Aug |
| `is_peak_travel` | Int | — | Binary: Jun–Aug + Nov–Dec |
| `carrier_rolling_delay_3m` | Float | — | 3-month lagged avg delay rate per carrier |
| `carrier_rolling_delay_12m` | Float | — | 12-month lagged avg delay rate per carrier |
| `late_aircraft_roll_3m` | Float | — | 3-month lagged late aircraft rate per carrier |
| `airport_rolling_delay_3m` | Float | — | 3-month lagged avg delay rate per airport |
| `carrier_yoy_delay_change` | Float | — | Year-over-year delay rate delta per carrier |
| `delay_risk_label` | Int | — | **Target**: 0=on-time, 1=minor, 2=major, 3=high-risk |

**Class distribution (from EDA):** Class 0: 36.5% · Class 1: 32.9% · Class 2: 16.0% · Class 3: 14.5%  
**Imbalance ratio** (high-risk : on-time): 0.398 — apply class weighting in NB04.

In [15]:
# Save engineered features to S3 for downstream notebooks
s3_client = boto3.client('s3', region_name=region)
fg_df.to_csv('/tmp/engineered_features.csv', index=False)

try:
    s3_client.upload_file('/tmp/engineered_features.csv', PROCESSED_BUCKET, 'interim/engineered_features.csv')
    print(f'Saved to s3://{PROCESSED_BUCKET}/interim/engineered_features.csv')
except Exception as e:
    print(f'S3 upload note: {e}')

print(f'\nFeature Group: {FEATURE_GROUP_NAME}')
print(f'Total features: {len(FEATURE_COLS)}')
print(f'Total records ingested: {len(fg_df):,}')

Saved to s3://skyaware-processed-data/interim/engineered_features.csv

Feature Group: skyaware-flight-features
Total features: 33
Total records ingested: 281,032
